In [2]:
#layer 1

In [3]:
import pandas as pd

In [4]:
ratings = pd.read_csv('rating.csv')
movies = pd.read_csv('movie.csv')

In [5]:
print(ratings.shape)
print(movies.shape)
ratings.head()

(20000263, 4)
(27278, 3)


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [6]:
rating_sample = ratings.sample(n=100000, random_state=42)

print(rating_sample.shape)
print(rating_sample.head())

(100000, 4)
          userId  movieId  rating            timestamp
17679788  122270     8360     3.5  2012-04-22 01:07:04
7106385    49018       32     2.0  2001-09-11 07:50:36
12970708   89527   109374     3.5  2015-01-06 09:26:40
15426752  106704     1060     3.0  2000-01-22 21:27:57
6934678    47791     1732     2.0  2006-01-19 15:48:23


In [7]:
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 50].index
filtered_ratings = ratings[ratings['userId'].isin(active_users)]

print(filtered_ratings.shape)
print(filtered_ratings['userId'].nunique())

(18335758, 4)
85307


In [8]:
import numpy as np

np.random.seed(42)
selected_users = np.random.choice(active_users, size=2000, replace=False)
final_sample = filtered_ratings[filtered_ratings['userId'].isin(selected_users)]

print(final_sample.shape)
print(final_sample['userId'].nunique())
print(final_sample['movieId'].nunique())

(436986, 4)
2000
13040


In [9]:
n_users = final_sample['userId'].nunique()
n_movies = final_sample['movieId'].nunique()
n_ratings = len(final_sample)

sparsity = 1 - (n_ratings / (n_users * n_movies))
print("Sparsity: {:.2f}%".format(sparsity * 100))

Sparsity: 98.32%


In [10]:
final_sample.to_csv('final_sample.csv', index=False)

In [11]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(final_sample[['userId', 'movieId', 'rating']], reader)

In [12]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVD(n_factors=50, n_epochs=20, random_state=42)
model.fit(trainset)

In [13]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.8399
MAE:  0.6434


In [14]:
#layer 2 


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies['genres'] = movies['genres'].fillna('')

tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
genre_matrix = tfidf.fit_transform(movies['genres'])

print(genre_matrix.shape)

(27278, 20)


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

print(cosine_sim.shape)

(27278, 27278)


In [17]:
movies = movies.reset_index(drop=True)
indices = pd.Series(movies.index, index=movies['title'])

def get_similar_movies(title, top_n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]

print(get_similar_movies('Toy Story (1995)'))

2209                                       Antz (1998)
3027                                Toy Story 2 (1999)
3663    Adventures of Rocky and Bullwinkle, The (2000)
3922                  Emperor's New Groove, The (2000)
4790                             Monsters, Inc. (2001)
Name: title, dtype: object


In [18]:
#layer 3 

In [19]:
def hybrid_recommend(user_id, movie_title=None, top_n=5):
    user_ratings_count = final_sample[final_sample['userId'] == user_id].shape[0]
    
    print(f"User {user_id} has {user_ratings_count} ratings in our data.")
    
    if user_ratings_count < 5:
        print("-> Cold-start case: using Content-Based Filtering")
        if movie_title is None:
            return "Naya user hai aur koi movie reference nahi di - content-based ke liye ek movie title chahiye"
        return get_similar_movies(movie_title, top_n)
    else:
        print("-> Enough history: using Collaborative Filtering (SVD)")
    
        watched = final_sample[final_sample['userId'] == user_id]['movieId'].tolist()
        all_movie_ids = movies['movieId'].unique()
        unwatched = [m for m in all_movie_ids if m not in watched]
        
        predictions = [(m, model.predict(user_id, m).est) for m in unwatched[:2000]]
        predictions.sort(key=lambda x: x[1], reverse=True)
        top_movies = predictions[:top_n]
        
        result = []
        for movie_id, pred_rating in top_movies:
            title = movies[movies['movieId'] == movie_id]['title'].values[0]
            result.append((title, round(pred_rating, 2)))
        return result

In [20]:
print(hybrid_recommend(user_id=999999999, movie_title='Toy Story (1995)'))

User 999999999 has 0 ratings in our data.
-> Cold-start case: using Content-Based Filtering
2209                                       Antz (1998)
3027                                Toy Story 2 (1999)
3663    Adventures of Rocky and Bullwinkle, The (2000)
3922                  Emperor's New Groove, The (2000)
4790                             Monsters, Inc. (2001)
Name: title, dtype: object


In [21]:
sample_user = final_sample['userId'].iloc[0]
print("Testing with user:", sample_user)

print(hybrid_recommend(user_id=sample_user))

Testing with user: 358
User 358 has 104 ratings in our data.
-> Enough history: using Collaborative Filtering (SVD)
[('Rear Window (1954)', np.float64(3.93)), ('Jean de Florette (1986)', np.float64(3.91)), ('Kolya (Kolja) (1996)', np.float64(3.81)), ('Vertigo (1958)', np.float64(3.74)), ('Out of the Past (1947)', np.float64(3.74))]


In [22]:
from collections import defaultdict

def precision_at_k(predictions, k=5, threshold=3.5):
    user_est_true = defaultdict(list)
    for pred in predictions:
        user_est_true[pred.uid].append((pred.est, pred.r_ui))

    precisions = {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]
        
        n_relevant_and_recommended = sum((true_r >= threshold) for (_, true_r) in top_k)
        precisions[uid] = n_relevant_and_recommended / k

    return sum(prec for prec in precisions.values()) / len(precisions)

precision = precision_at_k(predictions, k=5, threshold=3.5)
print(f"Precision@5: {precision:.4f}")

Precision@5: 0.8252


In [23]:
import pickle

# Model save karo
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Cosine similarity save karo
with open('cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

# Movies aur final_sample bhi CSV mein save kar (agar pehle nahi kiya)
movies.to_csv('movies_clean.csv', index=False)
final_sample.to_csv('final_sample.csv', index=False)

In [24]:
import pickle

with open('genre_matrix.pkl', 'wb') as f:
    pickle.dump(genre_matrix, f)